# Train Chess Figurine Classifier (HOG + MLP → TFLite)

Train a 5-class classifier to recognize chess piece figurines (K, Q, R, B, N)
using **HOG (Histogram of Oriented Gradients) features** fed into a small **Keras MLP**,
exported to TFLite.

**Why HOG instead of raw pixels?**
HOG captures gradient structure (edges, shapes) that distinguishes K/Q/R/B/N regardless
of font, DPI, or rendering style — much more robust than pixel-level CNN features.

**Why a custom HOG (not scikit-image)?**
The HOG computation mirrors `HogExtractor.extract()` in Flutter **exactly** —
no rounding differences, no library version drift.

**Why ±35° rotation augmentation?**
PDF pages are sometimes scanned at a slight angle (up to ~30°). Augmenting with ±35°
makes the model robust to tilted glyphs.

**Input:** `chess_glyphs_classifier.zip` from `extract_chess_glyphs`, uploaded to Drive.  
**Output:** `figurine_classifier.tflite` — model input is a 1767-float HOG vector.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## Step 2 — Copy zip from Drive and extract locally

Upload `chess_glyphs_classifier.zip` (produced by `extract_chess_glyphs`) to Google Drive
**as a single zip file** — do not unzip it on your machine first.  
Set `ZIP_ON_DRIVE` to its path, then run this cell.  
The zip is copied to Colab's local disk and extracted there for fast I/O during training.

In [ ]:
import os, shutil, zipfile

# ── EDIT THIS: path to the zip on your Drive ──────────────────────────────
ZIP_ON_DRIVE = '/content/gdrive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip'

LOCAL_ZIP    = '/content/chess_glyphs_classifier.zip'
EXTRACT_DIR  = '/content/glyphs_extracted'

if not os.path.exists(ZIP_ON_DRIVE):
    raise FileNotFoundError(
        f'Zip not found on Drive: {ZIP_ON_DRIVE}\n'
        f'Upload chess_glyphs_classifier.zip to that location first.'
    )

# Copy zip to local disk (fast — stays on Colab's own storage)
print('Copying zip from Drive to local disk...')
shutil.copy2(ZIP_ON_DRIVE, LOCAL_ZIP)
size_mb = os.path.getsize(LOCAL_ZIP) / (1024 * 1024)
print(f'  Copied: {size_mb:.1f} MB')

# Extract
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
print('Extracting...')
with zipfile.ZipFile(LOCAL_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Locate glyphs/ folder — search in root and one level down
GLYPHS_DIR = None
for candidate in [
    os.path.join(EXTRACT_DIR, 'glyphs'),
    os.path.join(EXTRACT_DIR, 'glyphs_results', 'glyphs'),
]:
    if os.path.exists(candidate):
        GLYPHS_DIR = candidate
        break

if GLYPHS_DIR is None:
    raise FileNotFoundError(
        f'Could not locate glyphs/ folder inside the zip.\n'
        f'Top-level contents: {os.listdir(EXTRACT_DIR)}'
    )

print(f'\n✅ Glyphs extracted to {GLYPHS_DIR}')
from PIL import Image

def count_images(folder):
    count = 0
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if not os.path.isfile(path):
            continue
        try:
            Image.open(path).verify()
            count += 1
        except Exception:
            pass
    return count

total = 0
for piece in sorted(os.listdir(GLYPHS_DIR)):
    piece_dir = os.path.join(GLYPHS_DIR, piece)
    if not os.path.isdir(piece_dir):
        continue
    n = count_images(piece_dir)
    total += n
    print(f'  {"✅" if n > 0 else "❌"} {piece}: {n} images')
print(f'  Total: {total} images')

## Step 3 — Install dependencies

In [ ]:
!pip install -q tensorflow pillow numpy scikit-image scikit-learn matplotlib

## Step 4 — Configuration and HOG feature extractor

**`_compute_hog_features` is the reference implementation.**  
It mirrors `HogExtractor.extract()` in `lib/chess/hog_extractor.dart` line-for-line.  
Do **not** replace this with `skimage.feature.hog` — scikit-image's internals differ
subtly and would break training/inference consistency.

In [ ]:
import gc
import ctypes
import random
import numpy as np
from PIL import Image, ImageOps, ImageFilter

CLASS_NAMES           = ['K', 'Q', 'R', 'B', 'N', 'NotAFigurine']  # 6th class = reject; put crops in glyphs/NotAFigurine/
IMG_SIZE              = 32
MAX_ROTATION_DEG      = 35
TARGET_TOTAL_SAMPLES  = 150_000
EPOCHS                = 80
BATCH_SIZE            = 128
VAL_SPLIT             = 0.15
TFLITE_PATH           = 'figurine_classifier.tflite'

# HOG constants — must stay in sync with hog_extractor.dart
_ORIENTATIONS = 9
_PX_PER_CELL  = 4
_CPB          = 2
_N_CELLS      = IMG_SIZE // _PX_PER_CELL          # 8
_N_BLOCKS     = _N_CELLS - _CPB + 1               # 7
_BLOCK_SIZE   = _CPB * _CPB * _ORIENTATIONS       # 36
FEATURE_DIM   = _N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE + 3  # 1767

# Precompute pixel→cell mapping once
_CY   = np.repeat(np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_CX   = np.tile(  np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_BASE = (_CY * _N_CELLS + _CX) * _ORIENTATIONS

# malloc_trim: tell glibc to return freed memory to the OS (Linux/Colab only)
try:
    _libc = ctypes.CDLL('libc.so.6')
    _has_malloc_trim = True
    print('malloc_trim available ✅')
except Exception:
    _has_malloc_trim = False
    print('malloc_trim not available (non-Linux), GC only')


def _malloc_trim():
    gc.collect()
    if _has_malloc_trim:
        _libc.malloc_trim(0)


def _compute_hog_features(img_32x32: np.ndarray) -> np.ndarray:
    """Vectorized HOG — matches Dart HogExtractor.extract() exactly."""
    gx = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gy = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gx[:, 1:-1] = img_32x32[:, 2:] - img_32x32[:, :-2]
    gy[1:-1, :] = img_32x32[2:, :] - img_32x32[:-2, :]

    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0

    bin_width = 180.0 / _ORIENTATIONS
    bf = ang.ravel() / bin_width
    b0 = bf.astype(np.int32) % _ORIENTATIONS
    b1 = (b0 + 1) % _ORIENTATIONS
    t  = bf - b0
    mf = mag.ravel()

    flat = np.bincount(
        np.concatenate([_BASE + b0, _BASE + b1]),
        weights=np.concatenate([mf * (1.0 - t), mf * t]),
        minlength=_N_CELLS * _N_CELLS * _ORIENTATIONS,
    )
    cell_hists = flat.reshape(_N_CELLS, _N_CELLS, _ORIENTATIONS)

    eps2    = 1e-5 ** 2
    hog_out = np.empty(_N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE, dtype=np.float64)
    out_i   = 0
    for by in range(_N_BLOCKS):
        for bx in range(_N_BLOCKS):
            block = cell_hists[by:by + _CPB, bx:bx + _CPB, :].ravel().copy()
            block /= np.sqrt(np.dot(block, block) + eps2)
            np.clip(block, 0.0, 0.2, out=block)
            block /= np.sqrt(np.dot(block, block) + eps2)
            hog_out[out_i:out_i + _BLOCK_SIZE] = block
            out_i += _BLOCK_SIZE
    return hog_out


def extract_features(arr_32x32: np.ndarray, orig_w: int, orig_h: int) -> np.ndarray:
    """
    HOG + shape stats from a pre-resized 32×32 float32 numpy array.
    No PIL, no skimage — zero extra allocation beyond the HOG itself.
    orig_w / orig_h: original glyph pixel dimensions (for aspect ratio feature).
    """
    img_gray = arr_32x32.astype(np.float64)
    hog_feat = _compute_hog_features(img_gray)
    extra = np.array([
        orig_w / max(orig_h, 1),
        np.mean(img_gray),
        np.std(img_gray),
    ], dtype=np.float64)
    return np.concatenate([hog_feat, extra]).astype(np.float32)


# Sanity-check
_dummy = extract_features(np.full((IMG_SIZE, IMG_SIZE), 0.5, dtype=np.float32), 32, 32)
assert len(_dummy) == FEATURE_DIM
del _dummy
mem_gb = TARGET_TOTAL_SAMPLES * FEATURE_DIM * 4 / 1e9
print(f'HOG feature vector : {FEATURE_DIM} dims  ✅')
print(f'Target samples     : {TARGET_TOTAL_SAMPLES:,}  (X ≈ {mem_gb:.2f} GB pre-allocated)')
print(f'Classes            : {len(CLASS_NAMES)}  {CLASS_NAMES}')

## Step 5 — Index labeled glyph images (paths only, no RAM cost)

In [ ]:
from collections import defaultdict

def index_glyphs(glyphs_dir, class_names):
    """Return (paths, labels, counts) — images are NOT loaded into RAM."""
    paths, labels, counts = [], [], defaultdict(int)
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(glyphs_dir, class_name)
        if not os.path.exists(class_dir):
            print(f'⚠️  {class_name} folder not found')
            continue
        for filename in sorted(os.listdir(class_dir)):
            filepath = os.path.join(class_dir, filename)
            if os.path.isfile(filepath) and filename.lower().endswith('.png'):
                paths.append(filepath)
                labels.append(class_idx)
                counts[class_name] += 1
    return paths, labels, counts

print(f'Indexing glyphs in {GLYPHS_DIR}...\n')
paths, labels, counts = index_glyphs(GLYPHS_DIR, CLASS_NAMES)
print(f'✅ Indexed {len(paths)} glyph images (paths only — no RAM used)\n')
for name in CLASS_NAMES:
    n = counts[name]
    print(f'  {"✅" if n > 0 else "❌"} {name}: {n}')

## Step 6 — Data augmentation + HOG feature extraction

In [ ]:
def augment_to_array(pil_img, n, size=32):
    """
    Augment a PIL image n times with:
    - Rotation ±35°
    - Scale 0.80–1.20
    - Brightness ±20%
    - Horizontal flip
    - Gaussian noise
    Returns list of float32 (size, size) numpy arrays.
    """
    orig_w, orig_h = pil_img.width, pil_img.height
    base_pil = pil_img.convert('L').resize((size, size), Image.LANCZOS)
    base = np.array(base_pil, dtype=np.float32) / 255.0
    del base_pil

    results = []
    for _ in range(n):
        arr = base.copy()

        # Brightness adjustment
        brightness = random.uniform(0.8, 1.2)
        arr = np.clip(arr * brightness, 0.0, 1.0).astype(np.float32)

        # Rotation
        pil = Image.fromarray((arr * 255).astype(np.uint8))
        pil = pil.rotate(random.uniform(-MAX_ROTATION_DEG, MAX_ROTATION_DEG),
                         resample=Image.BICUBIC, fillcolor=255)
        arr = np.array(pil, dtype=np.float32) / 255.0
        del pil

        # Scale
        scale    = random.uniform(0.80, 1.20)
        new_size = max(4, int(size * scale))
        pil2     = Image.fromarray((arr * 255).astype(np.uint8)).resize(
                       (new_size, new_size), Image.LANCZOS)
        small    = np.array(pil2, dtype=np.float32) / 255.0
        del pil2
        canvas   = np.ones((size, size), dtype=np.float32)
        off      = (size - new_size) // 2
        sy, sx   = max(0, off), max(0, off)
        ey       = min(sy + small.shape[0], size)
        ex       = min(sx + small.shape[1], size)
        canvas[sy:ey, sx:ex] = small[:ey - sy, :ex - sx]
        arr      = canvas

        # Flip
        if random.random() > 0.5:
            arr = arr[:, ::-1].copy()

        # Noise
        arr += np.random.normal(0, 0.03, arr.shape).astype(np.float32)
        results.append((np.clip(arr, 0.0, 1.0).astype(np.float32), orig_w, orig_h))
    return results


# ─────────────────────────────────────────────────────────────────────────
# Per-class augmentation: generate limited augmentations, then sample uniformly
# ─────────────────────────────────────────────────────────────────────────

AUGMENT_TOTAL_PER_CLASS  = 5000     # total augmentations to generate per class
KEEP_PER_CLASS           = 10000    # final dataset size per class

print(f'\n{"="*70}')
print('DATA AUGMENTATION (BALANCED)')
print(f'{"="*70}')
print(f'Per-class target  : {KEEP_PER_CLASS} images')
print(f'Augmentations gen : {AUGMENT_TOTAL_PER_CLASS} per class\n')

X_all, y_all = [], []

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_paths = [p for p, l in zip(paths, labels) if l == class_idx]
    
    if not class_paths:
        print(f'⚠️  {class_name}: no images found')
        continue
    
    print(f'{class_name}: {len(class_paths)} originals')
    
    # Load originals and extract HOG features
    X_orig, y_orig = [], []
    for i, filepath in enumerate(class_paths):
        with Image.open(filepath) as img:
            arr = np.array(img.convert('L').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS), dtype=np.float32) / 255.0
        X_orig.append(extract_features(arr, img.width, img.height))
        y_orig.append(class_idx)
        
        if (i + 1) % max(1, len(class_paths) // 5) == 0:
            pct = 100 * (i + 1) // len(class_paths)
            print(f'  Originals: {pct:3d}%', flush=True)
    
    X_orig = np.array(X_orig, dtype=np.float32)
    
    # Generate augmentations: sample images randomly, generate N variations of each
    num_to_augment = min(AUGMENT_TOTAL_PER_CLASS, len(class_paths) * 50)  # cap at 50 per image
    augment_per_image = max(1, num_to_augment // len(class_paths))
    
    X_aug, y_aug = [], []
    sampled_indices = random.sample(range(len(class_paths)), min(len(class_paths), AUGMENT_TOTAL_PER_CLASS // augment_per_image))
    
    aug_count = 0
    for idx, img_idx in enumerate(sampled_indices):
        filepath = class_paths[img_idx]
        with Image.open(filepath) as img:
            augs = augment_to_array(img, augment_per_image)
        
        for arr, orig_w, orig_h in augs:
            X_aug.append(extract_features(arr, orig_w, orig_h))
            y_aug.append(class_idx)
            aug_count += 1
        
        if (idx + 1) % max(1, len(sampled_indices) // 5) == 0:
            pct = 100 * (idx + 1) // len(sampled_indices)
            print(f'  Augmented: {pct:3d}% ({aug_count} samples)', flush=True)
    
    X_aug = np.array(X_aug, dtype=np.float32) if X_aug else np.empty((0, FEATURE_DIM), dtype=np.float32)
    
    # Combine originals + augmented
    X_combined = np.vstack([X_orig, X_aug])
    y_combined = np.array(y_orig + y_aug)
    
    # Shuffle
    perm = np.random.permutation(len(X_combined))
    X_combined = X_combined[perm]
    y_combined = y_combined[perm]
    
    # Keep first KEEP_PER_CLASS
    keep_count = min(KEEP_PER_CLASS, len(X_combined))
    X_combined = X_combined[:keep_count]
    y_combined = y_combined[:keep_count]
    
    X_all.append(X_combined)
    y_all.extend(y_combined)
    
    print(f'  → {len(X_orig)} orig + {len(X_aug)} aug = {len(X_combined)} total (kept {keep_count})\n')
    _malloc_trim()

# Concatenate all classes
X = np.vstack(X_all)
y = np.array(y_all)

# Final shuffle
perm = np.random.permutation(len(X))
X = X[perm]
y = y[perm]

print(f'\n✅ Final dataset: {len(X):,} samples  X: {X.shape}  ({X.nbytes / 1e9:.2f} GB)')
for class_idx, class_name in enumerate(CLASS_NAMES):
    count = np.sum(y == class_idx)
    print(f'  {class_name}: {count:,}')

## Step 7 — Train MLP on HOG features

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(FEATURE_DIM,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
], name='figurine_hog_mlp')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=12, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=6, min_lr=1e-6, monitor='val_accuracy'),
    ],
    verbose=1,
)
print(f'\n✅ Best val accuracy: {max(history.history["val_accuracy"]):.4%}')

## Step 8 — Per-class accuracy report

In [ ]:
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

y_pred_probs = model.predict(X_val, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

print('Per-class spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    probs = y_pred_probs[mask][0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy');  ax1.legend()
ax2.plot(history.history['loss'],         label='train')
ax2.plot(history.history['val_loss'],     label='val')
ax2.set_title('Loss');      ax2.legend()
plt.tight_layout(); plt.show()

## Step 9 — Export TFLite model

In [ ]:
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'✅ Saved: {TFLITE_PATH} ({size_kb:.0f} KB)')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f'   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')

print('\nTFLite spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    sample = X_val[mask][0:1].astype(np.float32)
    interp.set_tensor(inp['index'], sample)
    interp.invoke()
    probs = interp.get_tensor(out['index'])[0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

## Step 10 — Download model

In [ ]:
from google.colab import files
files.download(TFLITE_PATH)
print(f'✅ Downloaded {TFLITE_PATH}')